<a href="https://colab.research.google.com/github/clive-strath/SemProject_MLEngine/blob/main/CoinSprout_Step2_Returns_Covariance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CoinSprout: Step 2, Estimating Returns and Risk for Each Asset

**Goal:** produce the numbers the allocation engine (Step 3) needs:

- `mu`: the expected annual return of each asset (Money Market Fund, T-Bills, Equities)
- `cov`: a covariance matrix, which says how risky each asset is and how the assets move together

**What this notebook does**
1. Loads your three data sources
2. Puts them on one common basis (monthly returns, same months)
3. Estimates `mu` and `cov`, with a stable covariance estimator
4. Measures how *uncertain* those estimates are (bootstrap), which feeds the stability test later
5. Tests whether a forecasting model beats a simple baseline
6. Saves everything to `asset_stats.json`

**Honest warning before you start:** these three assets are not equally easy to measure. MMF and T-Bill figures are *rates* (smooth, slow-moving). Equities are *prices* (noisy, and the NASI index ignores dividends). The notebook flags problems when it finds them, so read the printed warnings.

## 0. Setup

In [ ]:
!pip -q install datasets

In [ ]:
import warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.arima.model import ARIMA

SEED = 42
rng = np.random.default_rng(SEED)
ASSETS = ["MMF", "TBill", "Equity"]      # keep this order everywhere

## 1. Load the three datasets

### 1a. Money Market Funds (Hugging Face dataset you already used)
The cell below downloads it. If the config name is different, the first line prints the available ones, so copy the right one into `CONFIG_NAME`.

In [ ]:
from datasets import load_dataset, get_dataset_config_names

REPO = "ToKnow-ai/Kenyan-Collective-Investment-Schemes-Dataset"
print("Available configs:", get_dataset_config_names(REPO))

CONFIG_NAME = "Processed Investment Schemes Records"   # change if the line above shows a different name
ds = load_dataset(REPO, CONFIG_NAME)
split = list(ds.keys())[0]
mmf_raw = ds[split].to_pandas()
print(mmf_raw.shape)
mmf_raw.head()

README.md:   0%|          | 0.00/6.67k [00:00<?, ?B/s]

Available configs: ['Approved Collective Investment Schemes', 'Processed Investment Schemes Records', 'Raw Cytonn Reports']


Processed Investment Schemes Records/dat(…): reconstructing file:   0%|          |  0.00B /  310kB            

Processed Investment Schemes Records/dat(…): downloading bytes:           |  0.00B            

Generating data split:   0%|          | 0/86858 [00:00<?, ? examples/s]

(86858, 4)


,entry_type,entry_date,entry_value,entry_scheme
0,1,2017-11-01,11.1,Cytonn Unit Trust Scheme
1,0,2018-01-01,256.6,Apollo Unit Trust Scheme
2,1,2018-01-01,10.1,CIC Unit Trust Scheme
3,0,2018-01-01,19756.7,CIC Unit Trust Scheme
4,0,2018-01-01,5837.0,NCBA Unit Trust Funds


**Already have the MMF data as a CSV from `MMF_Exploration`?** Skip the cell above and use this instead (remove the `#` marks):

In [ ]:
# from google.colab import files
# up = files.upload()                       # choose your MMF csv
# mmf_raw = pd.read_csv(list(up.keys())[0])
# mmf_raw.head()

### 1b. Treasury Bill rates (CBK)
You need a CSV with a **date column** and the **91-day T-Bill rate column**. Get the historical rates from the Central Bank of Kenya website (search "CBK historical Treasury bill rates"), paste them into a spreadsheet, and save as CSV. Data can be weekly or monthly; the notebook averages it to months.

The 91-day bill is the best match for CoinSprout: short term and easy to explain.

In [ ]:
from google.colab import files
up = files.upload()                           # choose your T-Bill csv
tbill_raw = pd.read_csv(list(up.keys())[0])
print(tbill_raw.shape)
tbill_raw.head()

KeyboardInterrupt: 

### 1c. NSE equity data
This is the file from your document (`NSE_combination_data_all_stock_2013_2025.csv`).

In [ ]:
up = files.upload()                           # choose the NSE csv
nse_raw = pd.read_csv(list(up.keys())[0])
print(nse_raw.shape)
nse_raw.head()

### 1d. Tell the notebook which columns to use
Run the next cell. It lists the columns of each dataset. Then **set the names in the cell after it** if the defaults are wrong.

In [ ]:
for name, d in [("MMF", mmf_raw), ("T-Bill", tbill_raw), ("NSE", nse_raw)]:
    print(f"{name} columns:", list(d.columns)[:30])
print()
print("MMF entry types:", mmf_raw["entry_type"].unique()[:10] if "entry_type" in mmf_raw.columns else "no 'entry_type' column")

In [ ]:
# ---------- EDIT THESE if your column names are different ----------
MMF_DATE_COL   = "date"
MMF_VALUE_COL  = "value"
MMF_TYPE_COL   = "entry_type"
MMF_TYPE_VALUE = "EFFECTIVE_ANNUAL_RATE"

TBILL_DATE_COL = "Date"
TBILL_RATE_COL = "91-day"

NSE_DATE_COL   = "Date"
NSE_INDEX_COL  = "^NASI"

# NASI is a PRICE index and ignores dividends. If you want to add an assumed dividend yield,
# set it here (e.g. 0.06 for 6% a year) and cite a source in your report. Leave 0.0 to stay price-only.
EQUITY_DIVIDEND_YIELD = 0.0
# --------------------------------------------------------------------

def check(df, cols, label):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{label}: column(s) {missing} not found. Available: {list(df.columns)}")

check(mmf_raw,   [MMF_DATE_COL, MMF_VALUE_COL, MMF_TYPE_COL], "MMF")
check(tbill_raw, [TBILL_DATE_COL, TBILL_RATE_COL], "T-Bill")
check(nse_raw,   [NSE_DATE_COL, NSE_INDEX_COL], "NSE")
print("All columns found.")

## 2. Convert everything to monthly returns on one basis

Why monthly? The three sources have different frequencies (daily, weekly, irregular). Monthly is a common basis, with enough data points (about 100 to 140) for stable estimates.

How each asset becomes a monthly return:
- **MMF:** monthly average of the Effective Annual Rate across funds, converted to a monthly return with `(1 + rate)^(1/12) - 1`
- **T-Bill:** the same conversion on the monthly average 91-day rate
- **Equity:** month-end index level, then the percentage change month to month (plus dividend yield / 12 if you set one)

In [ ]:
def to_decimal_rate(s):
    # rates can be stored as 11.5 (percent) or 0.115 (decimal); detect and convert
    s = pd.to_numeric(s, errors="coerce")
    return s / 100 if s.dropna().median() > 1 else s

def monthly_series(df, date_col, val_col, how="mean"):
    d = df[[date_col, val_col]].copy()
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    d[val_col] = pd.to_numeric(d[val_col], errors="coerce")
    d = d.dropna()
    d["m"] = d[date_col].dt.to_period("M")
    g = d.sort_values(date_col).groupby("m")[val_col]
    return g.mean() if how == "mean" else g.last()

# ---- MMF ----
mmf = mmf_raw[mmf_raw[MMF_TYPE_COL] == MMF_TYPE_VALUE].copy()
mmf[MMF_VALUE_COL] = to_decimal_rate(mmf[MMF_VALUE_COL])
before = len(mmf)
mmf = mmf[(mmf[MMF_VALUE_COL] > 0) & (mmf[MMF_VALUE_COL] < 0.30)]      # drop obvious data errors
print(f"MMF rows kept: {len(mmf)} of {before} (removed {before-len(mmf)} outside 0% to 30%)")
mmf_rate = monthly_series(mmf, MMF_DATE_COL, MMF_VALUE_COL)

# ---- T-Bill ----
tb = tbill_raw.copy()
tb[TBILL_RATE_COL] = to_decimal_rate(tb[TBILL_RATE_COL])
tbill_rate = monthly_series(tb, TBILL_DATE_COL, TBILL_RATE_COL)

# ---- Equity ----
nse_level = monthly_series(nse_raw, NSE_DATE_COL, NSE_INDEX_COL, how="last")
equity_ret = nse_level.pct_change().dropna() + EQUITY_DIVIDEND_YIELD / 12

# ---- annual rate -> monthly return ----
mmf_ret   = (1 + mmf_rate) ** (1/12) - 1
tbill_ret = (1 + tbill_rate) ** (1/12) - 1

returns = pd.concat({"MMF": mmf_ret, "TBill": tbill_ret, "Equity": equity_ret}, axis=1).dropna()
returns = returns[ASSETS]
print(f"\nCommon period: {returns.index.min()} to {returns.index.max()}  ({len(returns)} months)")
returns.head()

### Look at the data
Check that nothing looks broken (sudden jumps, flat lines, wrong scale).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
pd.concat({"MMF": mmf_rate, "T-Bill 91d": tbill_rate}, axis=1).dropna().mul(100).plot(ax=ax[0])
ax[0].set_title("Annual rates (%)"); ax[0].set_ylabel("%")
nse_level.plot(ax=ax[1], title="NASI index level (month-end)")
plt.tight_layout(); plt.show()

## 3. Estimate `mu` (expected return) and `cov` (risk)

- **mu:** average monthly return x 12 (annualised)
- **cov:** monthly covariance x 12, estimated with **Ledoit-Wolf shrinkage**. With only about 100 months, a plain sample covariance is noisy, and Ledoit-Wolf pulls it toward a more stable version. This is a standard, defensible choice.

In [ ]:
mu_sample = returns.mean() * 12
cov_sample = returns.cov() * 12
lw = LedoitWolf().fit(returns.values)
cov_lw = pd.DataFrame(lw.covariance_ * 12, index=ASSETS, columns=ASSETS)

summary = pd.DataFrame({
    "Expected return (mu)": (mu_sample * 100).round(2),
    "Volatility": (np.sqrt(np.diag(cov_lw)) * 100).round(2),
})
summary.columns = ["Expected return % / yr", "Volatility % / yr"]
print("Shrinkage amount used by Ledoit-Wolf:", round(lw.shrinkage_, 3))
display(summary)

print("Correlation between assets:")
display(returns.corr().round(2))

### Sanity flags
These check for the problems that most often make portfolio optimisers misbehave.

In [ ]:
mu = mu_sample.copy()
corr = returns.corr()

if len(returns) < 60:
    print(f"WARNING: only {len(returns)} common months. Estimates will be unstable.")
if mu["Equity"] <= mu["MMF"]:
    print("WARNING: equities have a LOWER expected return than the money market fund over this period.")
    print("   The optimiser will then give equities only the minimum allowed weight to everyone.")
    print("   Common reasons: NASI ignores dividends, and the period includes a weak market.")
    print("   Options: add a sourced dividend yield (EQUITY_DIVIDEND_YIELD above), or state this as a finding.")
if corr.loc["MMF", "TBill"] > 0.8:
    print(f"NOTE: MMF and T-Bills are highly correlated ({corr.loc['MMF','TBill']:.2f}). Expected: MMFs mostly hold T-Bills.")
    print("   The optimiser will see them as near-substitutes and may swing between them.")
if mu["TBill"] > mu["MMF"]:
    print("NOTE: T-Bills out-earned the MMF on average (MMFs charge fees). Fine, but be able to explain it.")
print("Checks finished.")

## 4. How uncertain are these numbers? (bootstrap)

With about 10 years of data, `mu` is not known precisely, especially for equities. We estimate the uncertainty by re-sampling the months 200 times (in blocks of 6 months, to keep the time-pattern of rates). Each re-sample gives a slightly different `mu` and `cov`.

Two uses:
1. **Confidence ranges** to report honestly
2. **The stability test in Step 3:** run the allocator on all 200 versions and check that the splits stay similar

In [ ]:
B, BLOCK = 200, 6
X = returns.values
n = len(X)

def block_bootstrap_indices(n, block):
    n_blocks = int(np.ceil(n / block))
    starts = rng.integers(0, n, n_blocks)
    idx = np.concatenate([(s + np.arange(block)) % n for s in starts])[:n]   # circular blocks
    return idx

boot_mu, boot_cov = [], []
for _ in range(B):
    s = X[block_bootstrap_indices(n, BLOCK)]
    boot_mu.append(s.mean(axis=0) * 12)
    boot_cov.append(LedoitWolf().fit(s).covariance_ * 12)
boot_mu, boot_cov = np.array(boot_mu), np.array(boot_cov)

ci = pd.DataFrame({
    "mu (point estimate) %": (mu * 100).round(2),
    "95% range low %": (np.percentile(boot_mu, 2.5, axis=0) * 100).round(2),
    "95% range high %": (np.percentile(boot_mu, 97.5, axis=0) * 100).round(2),
}, index=ASSETS)
ci

**How to read this:** if the equity range is wide (for example from -8% to +12%), then any claim like "equities return X%" is weak. That is a real limitation, so state it. It is also exactly why the Step 3 stability test matters.

## 5. Does a forecasting model beat a simple baseline?

We test forecasts honestly: the data is split **in time order** (train on the past, test on the future, no shuffling), and every model is compared against naive baselines. The metric is MAE (average miss, lower is better).

- **Rates (MMF and T-Bill):** Last value, Mean so far, and AR(1), a simple ARIMA model
- **Equity returns:** Mean so far vs a linear regression on past returns

In [ ]:
def eval_rate_forecasts(series, test_frac=0.3):
    s = series.dropna().astype(float)
    n = len(s); start = int(n * (1 - test_frac))
    preds = {"Last value": [], "Mean so far": [], "AR(1)": []}
    actual = []
    for t in range(start, n):
        hist = s.iloc[:t].values
        actual.append(s.iloc[t])
        preds["Last value"].append(hist[-1])
        preds["Mean so far"].append(hist.mean())
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                fit = ARIMA(hist, order=(1, 0, 0)).fit()
                preds["AR(1)"].append(float(fit.forecast(1)[0]))
        except Exception:
            preds["AR(1)"].append(hist[-1])
    return pd.Series({k: mean_absolute_error(actual, v) for k, v in preds.items()})

mae_rates = pd.DataFrame({
    "MMF rate (MAE, % pts)": eval_rate_forecasts(mmf_rate) * 100,
    "T-Bill rate (MAE, % pts)": eval_rate_forecasts(tbill_rate) * 100,
}).round(3)
mae_rates

In [ ]:
def eval_equity_forecast(ret, test_frac=0.3):
    feat = pd.DataFrame({
        "lag1": ret.shift(1), "lag2": ret.shift(2),
        "roll3": ret.rolling(3).mean().shift(1),
    })
    d = pd.concat([feat, ret.rename("y")], axis=1).dropna()
    n = len(d); start = int(n * (1 - test_frac))
    pm, pl, act = [], [], []
    for t in range(start, n):
        tr, te = d.iloc[:t], d.iloc[t:t+1]
        pm.append(tr["y"].mean())
        pl.append(LinearRegression().fit(tr[["lag1","lag2","roll3"]], tr["y"]).predict(te[["lag1","lag2","roll3"]])[0])
        act.append(te["y"].iloc[0])
    return pd.Series({"Mean so far": mean_absolute_error(act, pm),
                      "Linear regression": mean_absolute_error(act, pl)})

mae_eq = (eval_equity_forecast(returns["Equity"]) * 100).round(3)
print("Equity monthly return forecast error (MAE, % pts):")
print(mae_eq.to_string())

for name, col in [("MMF", "MMF rate (MAE, % pts)"), ("T-Bill", "T-Bill rate (MAE, % pts)")]:
    print(f"Best for {name}: {mae_rates[col].idxmin()}")
print("Best for Equity:", mae_eq.idxmin())

**How to read this**
- If **Last value** or **Mean so far** wins, that is a valid finding: these series are hard to forecast, and your allocator correctly relies on historical estimates instead of a forecast.
- For equities, the regression most likely will *not* beat the mean. Report that plainly. It supports your point that CoinSprout's value comes from personalised guidance and trust, not from market prediction.
- If AR(1) clearly wins on the rate series, you can mention it as a modest, justified use of a forecasting model.

## 6. Save the numbers for Step 3

In [ ]:
payload = {
    "assets": ASSETS,
    "period": [str(returns.index.min()), str(returns.index.max())],
    "n_months": int(len(returns)),
    "equity_dividend_yield_assumed": EQUITY_DIVIDEND_YIELD,
    "mu": mu.round(6).tolist(),
    "cov": cov_lw.round(8).values.tolist(),
    "cov_sample": cov_sample.round(8).values.tolist(),
    "bootstrap_mu": boot_mu.round(6).tolist(),
    "bootstrap_cov": boot_cov.round(8).tolist(),
}
with open("asset_stats.json", "w") as f:
    json.dump(payload, f)
returns.to_csv("monthly_returns.csv")
print("Saved asset_stats.json and monthly_returns.csv")

In [ ]:
try:
    from google.colab import files
    files.download("asset_stats.json")
    files.download("monthly_returns.csv")
except ImportError:
    print("Not running in Colab. Files are in the current folder.")

## What to write in your report

- **Data:** three sources (MMF Effective Annual Rates, CBK 91-day T-Bill rates, NASI), converted to monthly returns over the common period shown in Section 2.
- **Estimation:** annualised mean returns, plus a Ledoit-Wolf shrinkage covariance matrix, which is stable on small samples.
- **Uncertainty:** 200 block-bootstrap re-samples (Section 4). Quote the 95% ranges.
- **Forecasting:** compared against naive baselines in time order (Section 5). Report the winner honestly, even if it is a baseline.
- **Limitations:** NASI is price-only, monthly samples are short, and MMF and T-Bills are highly correlated.

## Next step
Step 3: the allocation engine. It reads `asset_stats.json`, takes a risk category from your classifier, and produces the percentage split, then runs the stability test on all 200 bootstrap versions.